# AURA – Investigation Engine

This notebook converts the fraud detection and explainability pipeline into a reusable investigation engine.

Input:
- Transaction ID

Output:
- Fraud probability
- Risk score
- Risk band
- SHAP-based risk factors
- Rule-based investigation reasons
- Recommended action

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap
from src.graph.graph_analyzer import GraphAnalyzer



MODEL_PATH = Path("../models/random_forest_final.joblib")
DATA_PATH = Path("../data/processed/sparkov_features.parquet")

rf_final = joblib.load(MODEL_PATH)
df = pd.read_parquet(DATA_PATH)

print("Dataset:", df.shape)
print("Model:", type(rf_final).__name__)

Dataset: (1296675, 25)
Model: Pipeline


In [2]:
MODEL_FEATURES = [
    "amt",
    "log_amount",
    "hour",
    "day_of_week",
    "month",
    "day_of_month",
    "is_weekend",
    "customer_prev_count",
    "customer_prev_avg_amount",
    "customer_prev_median_amount",
    "customer_prev_std_amount",
    "amount_deviation_ratio",
    "seconds_since_prev_transaction",
    "transactions_prev_5min",
    "transactions_prev_1h",
    "transactions_prev_24h",
    "merchant_seen_before",
    "is_new_merchant",
    "category_seen_before",
    "is_new_category",
    "customer_merchant_distance_km"
]

FINAL_THRESHOLD = 0.22

In [ ]:
def investigate_transaction(transaction_id):

    # Find transaction
    matches = df[df["trans_num"] == transaction_id]

    if matches.empty:
        return {
            "error": f"Transaction {transaction_id} not found."
        }

    if len(matches) > 1:
        return {
            "error": f"Multiple records found for {transaction_id}."
        }

    row = matches.iloc[0]

    # Prepare model input
    X = row[MODEL_FEATURES].to_frame().T

    # Model probability
    fraud_probability = rf_final.predict_proba(X)[0, 1]
    risk_score = round(float(fraud_probability * 100), 2)

    # Risk band
    if fraud_probability >= 0.50:
        risk_band = "HIGH"
        action = "ESCALATE FOR INVESTIGATION"
    elif fraud_probability >= FINAL_THRESHOLD:
        risk_band = "MEDIUM"
        action = "SEND TO REVIEW QUEUE"
    else:
        risk_band = "LOW"
        action = "APPROVE"

    # Rule-based reasons
    reasons = []

    if pd.notna(row["transactions_prev_1h"]) and row["transactions_prev_1h"] >= 3:
        reasons.append("High transaction velocity in the previous hour")

    if pd.notna(row["transactions_prev_5min"]) and row["transactions_prev_5min"] >= 2:
        reasons.append("Multiple transactions within a short time window")

    if pd.notna(row["amount_deviation_ratio"]) and row["amount_deviation_ratio"] >= 3:
        reasons.append("Transaction amount is substantially above customer history")

    if row["is_new_merchant"] == 1:
        reasons.append("Previously unseen merchant for this customer")

    if row["is_new_category"] == 1:
        reasons.append("Previously unseen transaction category")

    if (
        pd.notna(row["customer_merchant_distance_km"])
        and row["customer_merchant_distance_km"] >= 200
    ):
        reasons.append("Large customer-merchant geographic distance")

    if row["customer_prev_count"] == 0:
        reasons.append("No prior transaction history available")

    if not reasons:
        reasons.append("Model identified an anomalous transaction pattern")

    # SHAP explanation
    imputer = rf_final.named_steps["imputer"]
    rf_model = rf_final.named_steps["model"]

    X_imputed = imputer.transform(X)

    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_imputed)

    shap_values = np.asarray(shap_values)

    if shap_values.ndim == 3:
        shap_fraud = shap_values[0, :, 1]
    elif shap_values.ndim == 2:
        shap_fraud = shap_values[0]
    else:
        raise ValueError(
            f"Unexpected SHAP shape: {shap_values.shape}"
        )

    shap_table = pd.DataFrame({
        "feature": MODEL_FEATURES,
        "value": X.iloc[0].values,
        "shap_value": shap_fraud
    })

    shap_table["abs_shap"] = shap_table["shap_value"].abs()

    shap_table = shap_table.sort_values(
        "abs_shap",
        ascending=False
    )

    top_risk_factors = (
        shap_table[shap_table["shap_value"] > 0]
        .head(5)
        .to_dict("records")
    )

    return {
        "transaction_id": transaction_id,
        "transaction_time": str(row["transaction_time"]),
        "amount": float(row["amt"]),
        "fraud_probability": float(fraud_probability),
        "risk_score": risk_score,
        "risk_band": risk_band,
        "recommended_action": action,
        "investigation_reasons": reasons,
        "top_risk_factors": top_risk_factors
    }

In [4]:
transaction_id = "b991672c504a79471a46d05f02ca4109"

report = investigate_transaction(transaction_id)

report

{'transaction_id': 'b991672c504a79471a46d05f02ca4109',
 'transaction_time': '2020-04-03 10:59:44',
 'amount': 9.11,
 'fraud_probability': 1.0,
 'risk_score': 100.0,
 'risk_band': 'HIGH',
 'recommended_action': 'ESCALATE FOR INVESTIGATION',
 'investigation_reasons': ['Model identified an anomalous transaction pattern'],
 'top_risk_factors': [{'feature': 'amount_deviation_ratio',
   'value': 0.1345828368826214,
   'shap_value': 0.051617092143257735,
   'abs_shap': 0.051617092143257735},
  {'feature': 'log_amount',
   'value': 2.3135250330323798,
   'shap_value': 0.04619191242869194,
   'abs_shap': 0.04619191242869194},
  {'feature': 'amt',
   'value': 9.11,
   'shap_value': 0.04276156021109219,
   'abs_shap': 0.04276156021109219},
  {'feature': 'hour',
   'value': 10,
   'shap_value': 0.040466517800828015,
   'abs_shap': 0.040466517800828015},
  {'feature': 'category_seen_before',
   'value': 130,
   'shap_value': 0.03903113855915486,
   'abs_shap': 0.03903113855915486}]}

In [5]:
print("========================================")
print("         AURA INVESTIGATION REPORT")
print("========================================")

print(f"Transaction ID : {report['transaction_id']}")
print(f"Transaction Time: {report['transaction_time']}")
print(f"Amount         : {report['amount']}")
print(f"Fraud Probability: {report['fraud_probability']:.4f}")
print(f"Risk Score     : {report['risk_score']}/100")
print(f"Risk Band      : {report['risk_band']}")
print(f"Action         : {report['recommended_action']}")

print("\nInvestigation Reasons:")
for reason in report["investigation_reasons"]:
    print(f" - {reason}")

print("\nTop Model Risk Factors:")
for factor in report["top_risk_factors"]:
    print(
        f" - {factor['feature']}: "
        f"value={factor['value']}, "
        f"SHAP={factor['shap_value']:.4f}"
    )

         AURA INVESTIGATION REPORT
Transaction ID : b991672c504a79471a46d05f02ca4109
Transaction Time: 2020-04-03 10:59:44
Amount         : 9.11
Fraud Probability: 1.0000
Risk Score     : 100.0/100
Risk Band      : HIGH
Action         : ESCALATE FOR INVESTIGATION

Investigation Reasons:
 - Model identified an anomalous transaction pattern

Top Model Risk Factors:
 - amount_deviation_ratio: value=0.1345828368826214, SHAP=0.0516
 - log_amount: value=2.3135250330323798, SHAP=0.0462
 - amt: value=9.11, SHAP=0.0428
 - hour: value=10, SHAP=0.0405
 - category_seen_before: value=130, SHAP=0.0390


In [6]:
# Generate model scores for a sample of transactions
sample_df = df.sample(
    1000,
    random_state=42
).copy()

sample_df["fraud_probability"] = rf_final.predict_proba(
    sample_df[MODEL_FEATURES]
)[:, 1]

sample_df["risk_score"] = (
    sample_df["fraud_probability"] * 100
)

sample_df["risk_band"] = np.select(
    [
        sample_df["fraud_probability"] >= 0.50,
        sample_df["fraud_probability"] >= FINAL_THRESHOLD
    ],
    [
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

sample_df["risk_band"].value_counts()

risk_band
LOW       992
HIGH        7
MEDIUM      1
Name: count, dtype: int64

In [7]:
high_id = sample_df.loc[
    sample_df["risk_band"] == "HIGH",
    "trans_num"
].iloc[0]

medium_id = sample_df.loc[
    sample_df["risk_band"] == "MEDIUM",
    "trans_num"
].iloc[0]

low_id = sample_df.loc[
    sample_df["risk_band"] == "LOW",
    "trans_num"
].iloc[0]

print("HIGH:", high_id)
print("MEDIUM:", medium_id)
print("LOW:", low_id)

HIGH: f350a9c6b7146c67903f342e79fd32a1
MEDIUM: bc4dede08eb54572390a3a54c2a79064
LOW: 29660793e5ad40b37a640e19762021f7


In [8]:
for label, transaction_id in [
    ("HIGH", high_id),
    ("MEDIUM", medium_id),
    ("LOW", low_id)
]:
    print("\n" + "=" * 60)
    print(label, "CASE")
    print("=" * 60)

    report = investigate_transaction(transaction_id)

    print("Risk Score:", report.get("risk_score"))
    print("Risk Band:", report.get("risk_band"))
    print("Action:", report.get("recommended_action"))

    print("\nReasons:")
    for reason in report.get("investigation_reasons", []):
        print("-", reason)


HIGH CASE
Risk Score: 100.0
Risk Band: HIGH
Action: ESCALATE FOR INVESTIGATION

Reasons:
- Transaction amount is substantially above customer history
- Previously unseen merchant for this customer

MEDIUM CASE
Risk Score: 25.0
Risk Band: MEDIUM
Action: SEND TO REVIEW QUEUE

Reasons:
- Transaction amount is substantially above customer history
- Previously unseen merchant for this customer

LOW CASE
Risk Score: 0.0
Risk Band: LOW
Action: APPROVE

Reasons:
- Previously unseen merchant for this customer


In [10]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: c:\Users\HP\Desktop\AURA-Fraud-Intelligence


In [13]:
from src.investigation.investigation_engine import InvestigationEngine

engine = InvestigationEngine()

report = engine.investigate_transaction(
    "b991672c504a79471a46d05f02ca4109"
)

report

{'transaction_id': 'b991672c504a79471a46d05f02ca4109',
 'transaction_time': '2020-04-03 10:59:44',
 'amount': 9.11,
 'fraud_probability': 1.0,
 'risk_score': 100.0,
 'risk_band': 'HIGH',
 'recommended_action': 'ESCALATE FOR INVESTIGATION',
 'investigation_reasons': ['Model identified an anomalous transaction pattern'],
 'top_risk_factors': [{'feature': 'amount_deviation_ratio',
   'value': 0.1345828368826214,
   'shap_value': 0.051617092143257735,
   'abs_shap': 0.051617092143257735},
  {'feature': 'log_amount',
   'value': 2.3135250330323798,
   'shap_value': 0.04619191242869194,
   'abs_shap': 0.04619191242869194},
  {'feature': 'amt',
   'value': 9.11,
   'shap_value': 0.04276156021109219,
   'abs_shap': 0.04276156021109219},
  {'feature': 'hour',
   'value': 10,
   'shap_value': 0.040466517800828015,
   'abs_shap': 0.040466517800828015},
  {'feature': 'category_seen_before',
   'value': 130,
   'shap_value': 0.03903113855915486,
   'abs_shap': 0.03903113855915486}]}

In [14]:
test_transactions = {
    "HIGH": "f350a9c6b7146c67903f342e79fd32a1",
    "MEDIUM": "bc4dede08eb54572390a3a54c2a79064",
    "LOW": "29660793e5ad40b37a640e19762021f7"
}

for expected_band, transaction_id in test_transactions.items():

    report = engine.investigate_transaction(transaction_id)

    print("\n" + "=" * 60)
    print(f"EXPECTED: {expected_band}")
    print("=" * 60)

    print("Transaction:", transaction_id)
    print("Risk score:", report.get("risk_score"))
    print("Risk band:", report.get("risk_band"))
    print("Action:", report.get("recommended_action"))


EXPECTED: HIGH
Transaction: f350a9c6b7146c67903f342e79fd32a1
Risk score: 100.0
Risk band: HIGH
Action: ESCALATE FOR INVESTIGATION

EXPECTED: MEDIUM
Transaction: bc4dede08eb54572390a3a54c2a79064
Risk score: 25.0
Risk band: MEDIUM
Action: SEND TO REVIEW QUEUE

EXPECTED: LOW
Transaction: 29660793e5ad40b37a640e19762021f7
Risk score: 0.0
Risk band: LOW
Action: APPROVE
